# Model servers for the hijack-proof-agent eval

Serves the agent model (`qwen3.5:4b`) from three Ollama servers (two on GPU 0, one on GPU 1) and
the judge (`gemma4:12b-it-qat`) from a fourth on GPU 1, each behind a Cloudflare quick tunnel, and
prints the URLs the local eval runner calls. Ollama serves Qwen3.5 one request at a time
(the architecture is on its parallel-request blocklist; ollama/ollama#14510), so throughput comes from
several servers rather than parallel slots.

Models go to `/root` and logs to `/tmp`; neither counts toward the 19.5 GB output limit of
`/kaggle/working`, and `/root` is on the container disk rather than in memory. Start from a fresh session (Run → Restart session, or power off and on) so no
servers from an earlier run hold the ports.

Settings before **Run All**: Accelerator **GPU T4 x2**, Internet **on**. Keep the tab open; the last
cell keeps the session alive and prints a heartbeat. The tunnel URLs are random and change every
session; anyone holding one can call the model while the session runs.

In [ ]:
%%bash
set -uo pipefail
fail() { echo "FAILED: $1"; [ -n "${2:-}" ] && tail -40 "$2"; exit 1; }

curl -sSf -o /dev/null https://ollama.com || fail "no internet: Settings -> Internet -> On, then Run All again"
if ! command -v zstd > /dev/null; then
  (apt-get update -qq && apt-get install -y -qq zstd) > /tmp/apt.log 2>&1 || fail "installing zstd" /tmp/apt.log
fi
curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION=0.35.1 sh > /tmp/ollama-install.log 2>&1 || fail "installing ollama" /tmp/ollama-install.log
wget -q https://github.com/cloudflare/cloudflared/releases/download/2026.10.0/cloudflared-linux-amd64 \
  -O /usr/local/bin/cloudflared || fail "downloading cloudflared"
chmod +x /usr/local/bin/cloudflared
ollama --version; cloudflared --version
nvidia-smi --query-gpu=index,name,memory.total --format=csv

In [ ]:
import os, re, subprocess, time, json, urllib.request

AGENT, JUDGE = "qwen3.5:4b", "gemma4:12b-it-qat"
# The weights the reported runs used. Ollama pulls by tag only, so a tag that has moved since fails
# the check below instead of silently serving different weights.
WEIGHTS = {AGENT: "a9d0a3e1d8c91732a3812becd31f8d4a8ff57414903f226306069f0437e2ac50",
           JUDGE: "faff1a63667fac17ac5e777f47114688fcefea96e220e211aaa8d62c2c4561f1"}
SERVERS = {  # name: (gpu, port, model)
    "agent0": (0, 11434, AGENT),
    "agent1": (0, 11436, AGENT),
    "agent2": (1, 11437, AGENT),
    "judge": (1, 11435, JUDGE),
}
CONTEXT = 32768

def env_for(gpu, port, model):
    return {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "OLLAMA_HOST": f"127.0.0.1:{port}",
            "OLLAMA_MODELS": f"/root/ollama-{model.split(':')[0]}", "OLLAMA_CONTEXT_LENGTH": str(CONTEXT),
            "OLLAMA_NUM_PARALLEL": "1", "OLLAMA_MAX_LOADED_MODELS": "1", "OLLAMA_KEEP_ALIVE": "-1",
            "OLLAMA_NOPRUNE": "1"}  # servers sharing a model directory race each other when pruning at startup

procs = {}
for name, (gpu, port, model) in SERVERS.items():
    procs[name] = subprocess.Popen(["ollama", "serve"], env=env_for(gpu, port, model),
                                   stdout=open(f"/tmp/ollama-{name}.log", "w"), stderr=subprocess.STDOUT)
for name, (gpu, port, model) in SERVERS.items():  # wait until each server answers
    for _ in range(60):
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{port}/api/version", timeout=2); break
        except OSError:
            time.sleep(1)
    else:
        raise RuntimeError(f"{name}: ollama serve did not start on port {port}; see /tmp/ollama-{name}.log")
for name in ("agent0", "judge"):  # agent1 and agent2 share agent0's model directory
    gpu, port, model = SERVERS[name]
    subprocess.run(["ollama", "pull", model], env=env_for(gpu, port, model), check=True, stdout=subprocess.DEVNULL)
    blob = f"/root/ollama-{model.split(':')[0]}/blobs/sha256-{WEIGHTS[model]}"
    assert os.path.exists(blob), f"{model}: the tag now points to different weights than the reported runs used"
    print("pulled", model, "weights", WEIGHTS[model][:12])

In [ ]:
def chat(port, model, text):
    body = json.dumps({"model": model, "messages": [{"role": "user", "content": text}], "temperature": 0,
                       "max_tokens": 20, "reasoning_effort": "none"}).encode()
    req = urllib.request.Request(f"http://127.0.0.1:{port}/v1/chat/completions", body, {"Content-Type": "application/json"})
    return json.load(urllib.request.urlopen(req, timeout=600))

for name, (gpu, port, model) in SERVERS.items():
    r = chat(port, model, "Reply with the single word: ready")
    print(name, r["model"], repr(r["choices"][0]["message"]["content"]), r["usage"])
    print(json.load(urllib.request.urlopen(f"http://127.0.0.1:{port}/api/ps"))["models"][0]["size_vram"] // 2**20, "MiB in VRAM")
!nvidia-smi --query-gpu=index,memory.used,memory.total --format=csv

In [ ]:
urls = {}
for name, (gpu, port, model) in SERVERS.items():
    log = open(f"/tmp/tunnel-{name}.log", "w")
    procs[f"tunnel-{name}"] = subprocess.Popen(
        ["cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{port}",
         "--http-host-header", f"127.0.0.1:{port}"], stdout=log, stderr=subprocess.STDOUT)
for name in SERVERS:
    for _ in range(60):
        found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(f"/tmp/tunnel-{name}.log").read())
        if found:
            urls[name] = found.group(0); break
        time.sleep(2)
    else:
        raise RuntimeError(f"{name}: no tunnel URL after 120 s; see /tmp/tunnel-{name}.log")
open("/kaggle/working/server_urls.json", "w").write(json.dumps(urls))
print("Set these two variables when running evals/round.sh:\n")
print("AGENT_BASE_URLS=" + ",".join(f"{urls[n]}/v1" for n in SERVERS if n.startswith("agent")))
print(f"JUDGE_BASE_URL={urls['judge']}/v1")

In [ ]:
# Keeps the session alive; stop this cell (or the session) when the eval is done.
while True:
    used = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader"], capture_output=True, text=True).stdout.split()
    alive = {name: p.poll() is None for name, p in procs.items()}
    print(time.strftime("%H:%M:%S"), "GPU MiB used:", used, "alive:", alive, flush=True)
    time.sleep(300)